# Что внутри расплывчатых групп и сколько стоит отказ от защищённых признаков
Часть A: смотрим, из чего состоят группы причин, чтобы написать для них понятные формулировки.
Часть B: сравниваем модель с защищёнными признаками и без них, проверяем результаты по полу.
Всё на dev, hold-out не трогаем.

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

sys.path.insert(0, "..")
from src.data import load_features, split_dev_holdout
from src.explain import group_breakdown, shap_sample
from src.features import ID_COL, TARGET_COL
from src.metrics import report
from src.models import cv_predict, make_lgbm, to_lgbm

PROCESSED = Path("../data/processed")
MODELS = Path("../models")

df = load_features(PROCESSED / "train_features.parquet")
dev, _ = split_dev_holdout(df, PROCESSED / "holdout_ids.csv")  # hold-out отбрасываем
y = dev[TARGET_COL]
X_all = dev.drop(columns=[TARGET_COL, ID_COL])
cfg = json.loads((MODELS / "lgbm_best_params.json").read_text())

## A. Что внутри групп
Колонка spearman(value, SHAP): больше нуля, значит чем выше значение признака, тем выше риск;
меньше нуля, значит наоборот. Нужна, чтобы формулировка причины говорила в нужную сторону.

In [ ]:
model = joblib.load(MODELS / "lgbm_dev.joblib")
Xs, ys, sv, base, proba = shap_sample(model, to_lgbm(X_all), y, n=20_000, seed=42)  # та же выборка, что в 07

pd.set_option("display.width", 140)
for g in ["prev_other", "employment", "loan_terms", "bureau_active", "contacts"]:
    print(f"\n=== {g} ===")
    print(group_breakdown(sv, Xs, g, top=10))

## B1. Что делает модель с полом и семьёй
Сначала простая статистика по самим данным: как отличается дефолт по полу и что предсказывает модель.

In [ ]:
gender = X_all["CODE_GENDER"]
print("Доли в dev:")
print(gender.value_counts(dropna=False, normalize=True).round(3))
print("\nДоля дефолтов по полу:")
print(y.groupby(gender).agg(["mean", "count"]).round(4))

## B2. Цена отказа от защищённых признаков
Три набора признаков, одни и те же фолды и одни и те же гиперпараметры (число деревьев фиксировано):
1. full: как сейчас;
2. без пола, семейного положения, числа детей и членов семьи;
3. то же и без возраста (и без отношения стажа к возрасту, в котором зашит возраст).

In [ ]:
GENDER_FAMILY = ["CODE_GENDER", "NAME_FAMILY_STATUS", "CNT_CHILDREN", "CNT_FAM_MEMBERS", "INCOME_PER_PERSON"]
AGE_COLS = ["AGE", "DAYS_BIRTH", "EMPLOYED_BIRTH_RATIO"]

variants = {
    "full": X_all,
    "без пола и семьи": X_all.drop(columns=GENDER_FAMILY),
    "без пола, семьи и возраста": X_all.drop(columns=GENDER_FAMILY + AGE_COLS),
}


def make_model():
    return make_lgbm(n_estimators=cfg["n_estimators"], learning_rate=cfg["learning_rate"], **cfg["params"])


results = {}
for name, Xv in variants.items():
    oof, fold_auc = cv_predict(make_model, to_lgbm(Xv), y)
    results[name] = (oof, fold_auc)
    print(f"{name}: AUC по фолдам {fold_auc.mean():.4f} ± {fold_auc.std():.4f} | признаков: {Xv.shape[1]}")

In [ ]:
ref = results["full"][1]
rows = []
for name, (oof, fa) in results.items():
    d = fa - ref
    r = report(name, y, oof)
    r.update({"diff vs full (mean)": round(d.mean(), 4), "min": round(d.min(), 4), "max": round(d.max(), 4)})
    rows.append(r)
pd.DataFrame(rows).set_index("model")

## B3. Результаты по полу
Для каждого варианта (по OOF-предсказаниям):
- observed: реальная доля дефолтов;
- predicted: средняя предсказанная вероятность (если сильно расходится с observed, модель
  систематически занижает или завышает риск для группы);
- AUC внутри группы;
- flagged: доля клиентов в верхних 10% по риску (условный отказ).

In [ ]:
rows = []
for name, (oof, _) in results.items():
    thr = np.quantile(oof, 0.90)
    for g in ["F", "M"]:
        m = (gender == g).values
        rows.append({
            "вариант": name, "пол": g, "клиентов": int(m.sum()),
            "observed": y[m].mean(), "predicted": oof[m].mean(),
            "AUC в группе": roc_auc_score(y[m], oof[m]),
            "flagged, %": (oof[m] >= thr).mean() * 100,
        })
fair = pd.DataFrame(rows).round(4)
fair

In [ ]:
# Отношение долей отказов M / F: насколько различаются исходы для мужчин и женщин
ratio = fair.pivot(index="вариант", columns="пол", values="flagged, %")
ratio["M / F"] = (ratio["M"] / ratio["F"]).round(2)
ratio

Что смотрим:
1. `min diff vs full`: падает ли AUC при удалении признаков, и насколько стабильно;
2. сильно ли различается доля дефолтов по полу в самих данных (тогда модель отражает статистику выборки);
3. сокращается ли разрыв `flagged M / F` без пола и остаётся ли он за счёт косвенных признаков
   (ORGANIZATION_TYPE, OCCUPATION_TYPE и т.п.).